In [0]:
%run ../../02_common_utils/operations

In [0]:
from pyspark.sql.functions import *

In [0]:
# ─── CONFIG ──────────────────────────────────────────────────────────────
catalog             = "charles_schwab_retailbrokerage_dev_team_lemma"
bronze_customermgmt = f"{catalog}.bronze.customermgmt"   # B1: 50,000 XML actions
bronze_customer_txt = f"{catalog}.bronze.customer"        # B2/B3: 50 CDC rows each
silver_customer     = f"{catalog}.silver.customer"
silver_account      = f"{catalog}.silver.account"

In [0]:
# ─── STEP 1: Read bronze.customermgmt (B1 — XML) & register as temp view ──
# Schema has: ActionTS (string), C_PRIM_EMAIL, C_ALT_EMAIL, _batch_id
spark.read.table(bronze_customermgmt).createOrReplaceTempView("v_bronze_mgmt")
carried_run_id = spark.sql(f"SELECT _run_id FROM {bronze_customermgmt} LIMIT 1").first()[0]
carried_batch = spark.sql(f"SELECT _batch_id FROM {bronze_customermgmt} LIMIT 1").first()[0]
print(f"carried_run_id: {carried_run_id}")
print(f"carried_batch: {carried_batch}")
print(f"bronze.customermgmt rows: {spark.sql('SELECT COUNT(*) FROM v_bronze_mgmt').first()[0]}")

In [0]:
log_pipeline_message(spark, carried_run_id, 'INFO', 'silver_customer_customermgmt', 'Starting processing for silver customer and account')
start_pipeline_run(spark, carried_run_id, carried_batch)
log_domain_run_status(spark, carried_run_id, carried_batch, 'CUSTOMER', 'RUNNING')

In [0]:
# ─── STEP 2: Build customer rows from customermgmt (B1 NEW/UPDCUST/INACT) ─
# - ActionTS exists here → use as action_ts
# - Email cols are C_PRIM_EMAIL, C_ALT_EMAIL
# - Batch col is _batch_id

df_mgmt_customers = spark.sql("""
    SELECT
        CAST(ActionTS        AS TIMESTAMP)   AS action_ts,
        CAST(C_ID            AS BIGINT)      AS C_ID,
        C_TAX_ID,
        C_GNDR,
        TRY_CAST(C_TIER      AS TINYINT)     AS C_TIER,
        CAST(C_DOB           AS DATE)        AS C_DOB,
        C_L_NAME, C_F_NAME, C_M_NAME,
        C_ADLINE1, C_ADLINE2, C_ZIPCODE, C_CITY, C_STATE_PROV, C_CTRY,
        C_PRIM_EMAIL                         AS primary_email,   -- alias to common name
        C_ALT_EMAIL                          AS alternate_email,
        CONCAT_WS('-', NULLIF(C_CTRY_1,''), NULLIF(C_AREA_1,''), NULLIF(C_LOCAL_1,''), NULLIF(C_EXT_1,'')) AS phone1,
        CONCAT_WS('-', NULLIF(C_CTRY_2,''), NULLIF(C_AREA_2,''), NULLIF(C_LOCAL_2,''), NULLIF(C_EXT_2,'')) AS phone2,
        CONCAT_WS('-', NULLIF(C_CTRY_3,''), NULLIF(C_AREA_3,''), NULLIF(C_LOCAL_3,''), NULLIF(C_EXT_3,'')) AS phone3,
        C_LCL_TX_ID,
        C_NAT_TX_ID,
        _batch_id                            AS _batch,           -- rename to standard silver name
        _run_id,
        current_timestamp()                  AS _load_ts
    FROM v_bronze_mgmt
    WHERE ActionType IN ('NEW', 'UPDCUST', 'INACT')
""")
print(f"CustomerMgmt customer rows (B1): {df_mgmt_customers.count()}")

In [0]:
# ─── STEP 3: Read bronze.customer (B2/B3 — pipe-delimited CDC) ────────────
# Schema differences vs customermgmt:
#   - NO ActionTS  → use _ingest_ts as the action timestamp
#   - Email cols   → C_EMAIL_1, C_EMAIL_2 (not C_PRIM_EMAIL / C_ALT_EMAIL)
#   - Has CDC_FLAG, CDC_DSN, C_ST_ID (not present in customermgmt — ignore these)
#   - Batch col    → _batch_id (same as customermgmt)

spark.read.table(bronze_customer_txt).createOrReplaceTempView("v_bronze_cust_txt")
print(f"bronze.customer rows (B2+B3): {spark.sql('SELECT COUNT(*) FROM v_bronze_cust_txt').first()[0]}")

df_cust_txt = spark.sql("""
    SELECT
        CAST(_ingest_ts      AS TIMESTAMP)   AS action_ts,    -- NO ActionTS, use _ingest_ts instead
        CAST(C_ID            AS BIGINT)      AS C_ID,
        C_TAX_ID,
        C_GNDR,
        TRY_CAST(C_TIER      AS TINYINT)     AS C_TIER,
        CAST(C_DOB           AS DATE)        AS C_DOB,
        C_L_NAME, C_F_NAME, C_M_NAME,
        C_ADLINE1, C_ADLINE2, C_ZIPCODE, C_CITY, C_STATE_PROV, C_CTRY,
        C_EMAIL_1                            AS primary_email,  -- different name, same alias
        C_EMAIL_2                            AS alternate_email,
        CONCAT_WS('-', NULLIF(C_CTRY_1,''), NULLIF(C_AREA_1,''), NULLIF(C_LOCAL_1,''), NULLIF(C_EXT_1,'')) AS phone1,
        CONCAT_WS('-', NULLIF(C_CTRY_2,''), NULLIF(C_AREA_2,''), NULLIF(C_LOCAL_2,''), NULLIF(C_EXT_2,'')) AS phone2,
        CONCAT_WS('-', NULLIF(C_CTRY_3,''), NULLIF(C_AREA_3,''), NULLIF(C_LOCAL_3,''), NULLIF(C_EXT_3,'')) AS phone3,
        C_LCL_TX_ID,
        C_NAT_TX_ID,
        _batch_id                            AS _batch,
        _run_id,
        current_timestamp()                  AS _load_ts
    FROM v_bronze_cust_txt
""")
print(f"Customer.txt rows (B2+B3): {df_cust_txt.count()}")

In [0]:
# ─── STEP 4: UNION both sources into one dataset ──────────────────────────
# Both DataFrames now have IDENTICAL column names and types — safe to UNION
# This is required by the MD spec:
#   "UNION of CustomerMgmt.xml NEW actions (B1) + Customer.txt CDC (B2/B3),
#    dedup by C_ID (latest EffectiveDate wins)"

df_union = df_mgmt_customers.unionByName(df_cust_txt)
df_union.createOrReplaceTempView("v_all_customers")
print(f"Union total (before dedup): {df_union.count()}")

In [0]:
# ─── STEP 5: Deduplicate by C_ID — Latest action_ts wins ─────────────────
# B1 NEW rows → unique C_IDs → all survive (rn=1)
# B2 UPDATE rows (CDC_FLAG='U') → same C_ID exists in B1 → latest _ingest_ts wins → overwrites
# B2 INSERT rows (CDC_FLAG='I') → new C_IDs → inserted (rn=1)
# Result: count grows by 35 per batch (inserts only), never by 15 (updates overwrite)

df_customer_dedup = spark.sql("""
    SELECT * EXCEPT (rn)
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY C_ID
                ORDER BY action_ts DESC    -- Latest record wins (UPDATE overwrites INSERT)
            ) AS rn
        FROM v_all_customers
    )
    WHERE rn = 1
""")

# from pyspark.sql.window import Window
# from pyspark.sql.functions import row_number
# window_cust = Window.partitionBy("C_ID").orderBy(col("action_ts").desc())
# df_customer_dedup = df_union \
#     .withColumn("rn", row_number().over(window_cust)) \
#     .filter(col("rn") == 1).drop("rn")

df_customer_dedup.createOrReplaceTempView("v_customer_dedup")
customer_count = df_customer_dedup.count()
print(f"Deduplicated rows: {customer_count}")
# B1 only → 15,280 | After B2 → 15,315 | After B3 → 15,350

In [0]:
# ─── STEP 6: MERGE into silver.customer ──────────────────────────────────
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.silver;")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {silver_customer}
    USING DELTA
    AS SELECT * FROM v_customer_dedup WHERE 1=0
""")

# MERGE: MATCHED = UPDATE in place (no count increase), NOT MATCHED = INSERT (count +35)
spark.sql(f"""
    MERGE INTO {silver_customer} AS tgt
    USING v_customer_dedup       AS src
    ON tgt.C_ID = src.C_ID
    WHEN MATCHED THEN
        UPDATE SET *
    WHEN NOT MATCHED THEN
        INSERT *
""")

# from delta.tables import DeltaTable
# DeltaTable.forName(spark, silver_customer).alias("tgt") \
#     .merge(df_customer_dedup.alias("src"), "tgt.C_ID = src.C_ID") \
#     .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

silver_customer_count = spark.read.table(silver_customer).count()
print(f"silver.customer rows: {silver_customer_count}")

In [0]:
# ─── STEP 7: Build & Deduplicate silver.account (customermgmt B1 only) ───
# Account.txt (B2/B3) is handled separately by the Account domain notebook.
# Here we only push account actions from CustomerMgmt.xml.
#
# Target silver.account_copy schema:
#   CA_ID, CA_C_ID (customer FK), CA_B_ID, CA_NAME, CA_TAX_ST, CA_ST_ID, _batch, _run_id, _load_ts
#
# Fixes vs previous version:
#   - C_ID renamed to CA_C_ID  (was: 'C_ID', target expects 'CA_C_ID')
#   - CA_ST_ID ADDED — derived from ActionType (CLOSEACCT → 'INAC', others → 'ACTV')
#   - action_ts used for window dedup only, then DROPPED from final output

df_account = spark.sql("""
    SELECT
        CAST(ActionTS        AS TIMESTAMP)   AS action_ts,      -- for dedup ordering, dropped later
        CAST(CA_ID           AS BIGINT)      AS CA_ID,
        CAST(C_ID            AS BIGINT)      AS CA_C_ID,         -- FIX: renamed to match target schema
        CAST(CA_B_ID         AS BIGINT)      AS CA_B_ID,
        CA_NAME,
        TRY_CAST(CA_TAX_ST   AS TINYINT)     AS CA_TAX_ST,
        CASE ActionType
            WHEN 'CLOSEACCT' THEN 'INAC'                        -- FIX: CA_ST_ID derived from action
            ELSE 'ACTV'
        END                                  AS CA_ST_ID,
        _batch_id                            AS _batch,
        _run_id,
        current_timestamp()                  AS _load_ts
    FROM v_bronze_mgmt
    WHERE ActionType IN ('ADDACCT', 'UPDACCT', 'CLOSEACCT', 'NEW')
""")

df_account.createOrReplaceTempView("v_account_raw")

df_account_dedup = spark.sql("""
    SELECT * EXCEPT (rn, action_ts)    -- DROP action_ts: used for ordering only, not in target schema
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY CA_ID
                ORDER BY action_ts DESC
            ) AS rn
        FROM v_account_raw
    )
    WHERE rn = 1
""")

# ─── PySpark equivalent (commented for reference) ──────────────────────
# from pyspark.sql.window import Window
# from pyspark.sql.functions import row_number
# window_acct = Window.partitionBy("CA_ID").orderBy(col("action_ts").desc())
# df_account_dedup = df_account \
#     .withColumn("rn", row_number().over(window_acct)) \
#     .filter(col("rn") == 1).drop("rn", "action_ts")

df_account_dedup.createOrReplaceTempView("v_account_dedup")
account_count = df_account_dedup.count()
print(f"Account rows after dedup: {account_count}")

In [0]:
# ─── STEP 8: MERGE into silver.account ───────────────────────────────────
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {silver_account}
    USING DELTA
    AS SELECT * FROM v_account_dedup WHERE 1=0
""")

spark.sql(f"""
    MERGE INTO {silver_account} AS tgt
    USING v_account_dedup       AS src
    ON tgt.CA_ID = src.CA_ID
    WHEN MATCHED THEN
        UPDATE SET 
            tgt.CA_C_ID = src.CA_C_ID,
            tgt.CA_B_ID = src.CA_B_ID,
            tgt.CA_NAME = src.CA_NAME,
            tgt.CA_TAX_ST = src.CA_TAX_ST,
            tgt.CA_ST_ID = src.CA_ST_ID,
            tgt._batch = src._batch,
            tgt._run_id = src._run_id,
            tgt._load_ts = src._load_ts
    WHEN NOT MATCHED THEN
        INSERT (CA_ID, CA_C_ID, CA_B_ID, CA_NAME, CA_TAX_ST, CA_ST_ID, _batch, _run_id, _load_ts)
        VALUES (src.CA_ID, src.CA_C_ID, src.CA_B_ID, src.CA_NAME, src.CA_TAX_ST, src.CA_ST_ID, src._batch, src._run_id, src._load_ts)
""")

silver_account_count = spark.read.table(silver_account).count()
print(f"silver.account rows: {silver_account_count}")


In [0]:
# ─── STEP 9: Operations Audit Logging ────────────────────────────────────
log_pipeline_recon(
    spark=spark, run_id=carried_run_id, batch_id="ALL",
    domain="CUSTOMER", table_name="customer",
    source_layer="bronze", target_layer="silver",
    source_count=customer_count, target_count=silver_customer_count
)
log_audit_event(
    spark=spark, run_id=carried_run_id, batch="ALL",
    layer="silver", table_name="customer",
    operation="MERGE", rows_affected=silver_customer_count
)

log_pipeline_recon(
    spark=spark, run_id=carried_run_id, batch_id="ALL",
    domain="CUSTOMER", table_name="account",
    source_layer="bronze", target_layer="silver",
    source_count=account_count, target_count=silver_account_count
)
log_audit_event(
    spark=spark, run_id=carried_run_id, batch="ALL",
    layer="silver", table_name="account",
    operation="MERGE", rows_affected=silver_account_count
)



In [0]:
null_cid_count = spark.sql("SELECT COUNT(*) FROM v_customer_dedup WHERE C_ID IS NULL").first()[0]
log_dq_result(spark, carried_run_id, "silver.customer", "Null C_ID Check", null_cid_count, customer_count)

# 5. Pipeline Messages and End State
log_domain_run_status(spark, carried_run_id, carried_batch, 'CUSTOMER', 'COMPLETED')
end_pipeline_run(spark, carried_run_id, 'SUCCESS')
log_pipeline_message(spark, carried_run_id, 'INFO', 'silver_customer_customermgmt', 'Successfully completed processing customer and account domains')